In [1]:
import pandas as pd
import re

from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
from nltk.tokenize import word_tokenize

In [2]:
# Fungsi untuk mendeteksi bahasa
def detect_language(text):
    try:
        if pd.isna(text) or text.strip() == '':
            return 'unknown'
        return detect(text)
    except LangDetectException:
        return 'unknown'

In [3]:
# Fungsi untuk membersihkan teks dari emoji dan karakter khusus
def clean_text(text):
    if pd.isna(text):
        return ''

    # Hapus emoji
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               "]+", flags=re.UNICODE)

    text = emoji_pattern.sub(r'', text)

    # Hapus karakter khusus berlebihan
    text = re.sub(r'[^\w\s#@.,!?-]', ' ', text)

    return text.strip()

In [4]:
# Fungsi untuk mengecek apakah tweet hanya berisi hashtag, mention, atau link
def is_only_metadata(text):
    if pd.isna(text) or text.strip() == '':
        return True

    # Hapus hashtag, mention, dan link
    cleaned = re.sub(r'#\w+', '', text)  # hapus hashtag
    cleaned = re.sub(r'@\w+', '', cleaned)  # hapus mention
    cleaned = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', cleaned)  # hapus URL
    cleaned = re.sub(r'www\.(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', cleaned)  # hapus www

    # Cek apakah masih ada teks bermakna
    cleaned = cleaned.strip()
    return len(cleaned) < 3

In [5]:
# Fungsi untuk menghitung jumlah kata
def count_words(text):
    if pd.isna(text) or text.strip() == '':
        return 0

    words = word_tokenize(text.lower())
    meaningful_words = [word for word in words if word.isalnum() and len(word) > 1]

    return len(meaningful_words)

In [6]:
# Fungsi untuk mengecek apakah tweet mengandung kata "Rempang"
def contains_rempang(text):
    if pd.isna(text):
        return False

    return 'rempang' in text.lower()

In [7]:
# Fungsi utama untuk membersihkan data
def clean_dataset(df, text_column='tweet_text'):
    print(f"Data awal: {len(df)} tweets")

    cleaned_df = df.copy()

    # 1. Hapus tweet berbahasa Inggris
    print("1. Mendeteksi dan menghapus tweet berbahasa Inggris...")
    cleaned_df['language'] = cleaned_df[text_column].apply(detect_language)
    cleaned_df = cleaned_df[cleaned_df['language'] != 'en']
    print(f"   Sisa setelah filter bahasa: {len(cleaned_df)} tweets")

    # 2. Hapus tweet yang tidak memuat teks (kosong, hanya spasi, atau emoji)
    print("2. Menghapus tweet tanpa teks bermakna...")
    cleaned_df['cleaned_text'] = cleaned_df[text_column].apply(clean_text)
    cleaned_df = cleaned_df[cleaned_df['cleaned_text'].str.len() > 0]
    print(f"   Sisa setelah filter teks kosong: {len(cleaned_df)} tweets")

    # 3. Hapus tweet yang hanya berisi hashtag, mention, atau link
    print("3. Menghapus tweet yang hanya berisi metadata...")
    cleaned_df['is_metadata_only'] = cleaned_df['cleaned_text'].apply(is_only_metadata)
    cleaned_df = cleaned_df[~cleaned_df['is_metadata_only']]
    print(f"   Sisa setelah filter metadata: {len(cleaned_df)} tweets")

    # 4. Hapus tweet dengan kurang dari 25 kata
    print("4. Menghapus tweet dengan kurang dari 25 kata...")
    cleaned_df['word_count'] = cleaned_df['cleaned_text'].apply(count_words)
    cleaned_df = cleaned_df[cleaned_df['word_count'] >= 25]
    print(f"   Sisa setelah filter panjang: {len(cleaned_df)} tweets")

    # 6. Hapus tweet yang tidak menyebutkan "Rempang"
    print("6. Memfilter tweet yang mengandung kata 'Rempang'...")
    cleaned_df['contains_rempang'] = cleaned_df[text_column].apply(contains_rempang)
    cleaned_df = cleaned_df[cleaned_df['contains_rempang']]
    print(f"   Sisa setelah filter Rempang: {len(cleaned_df)} tweets")

    # 5. Hapus duplikasi (dilakukan setelah cleaning lainnya)
    print("5. Menghapus duplikasi...")
    initial_count = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates(subset=['cleaned_text'])
    print(f"   Duplikasi dihapus: {initial_count - len(cleaned_df)} tweets")
    print(f"   Sisa setelah hapus duplikasi: {len(cleaned_df)} tweets")

    return cleaned_df


In [8]:
# Baca dataset
print("Membaca dataset...")
df = pd.read_csv('../Meta/i18n/Dataset.csv')
print(f"Dataset loaded with {len(df)} tweets")

Membaca dataset...
Dataset loaded with 6079 tweets


In [9]:
# Jalankan pembersihan data
cleaned_data = clean_dataset(df, 'full_text')

Data awal: 6079 tweets
1. Mendeteksi dan menghapus tweet berbahasa Inggris...
   Sisa setelah filter bahasa: 6003 tweets
2. Menghapus tweet tanpa teks bermakna...
   Sisa setelah filter teks kosong: 5714 tweets
3. Menghapus tweet yang hanya berisi metadata...
   Sisa setelah filter metadata: 5546 tweets
4. Menghapus tweet dengan kurang dari 25 kata...
   Sisa setelah filter panjang: 1539 tweets
6. Memfilter tweet yang mengandung kata 'Rempang'...
   Sisa setelah filter Rempang: 1216 tweets
5. Menghapus duplikasi...
   Duplikasi dihapus: 8 tweets
   Sisa setelah hapus duplikasi: 1208 tweets


In [13]:
# Tampilkan hasil
print(f"\n=== HASIL PEMBERSIHAN DATA ===")
print(f"Data awal: {len(df)} tweets")
print(f"Data setelah dibersihkan: {len(cleaned_data)} tweets")


=== HASIL PEMBERSIHAN DATA ===
Data awal: 6079 tweets
Data setelah dibersihkan: 1208 tweets


In [11]:
# Jika Anda ingin mengambil tepat 1000 data (jika data yang bersih lebih dari 1000)
target_sample = 1000

if len(cleaned_data) > target_sample:
    print(f"\n=== SAMPLING DATA ===")
    print(f"Data tersedia: {len(cleaned_data)} tweets")
    print(f"Target sample: {target_sample} tweets")

    # Random sampling
    final_data = cleaned_data.sample(n=target_sample, random_state=42)
    print(f"Data final: {len(final_data)} tweets")

    # Simpan hasil
    final_data.to_csv('../Meta/DatasetBackup.csv', index=False)
    print("Data berhasil disimpan ke '../Meta/DatasetBackup.csv'")

elif len(cleaned_data) == 0:
    print("\n⚠️ PERINGATAN: Tidak ada data yang memenuhi semua kriteria filter!")
    print("Pertimbangkan untuk melonggarkan beberapa kriteria filter.")

else:
    print(f"\n=== DATA FINAL ===")
    print(f"Data final: {len(cleaned_data)} tweets (kurang dari target 1000)")
    final_data = cleaned_data

    # Simpan hasil
    final_data.to_csv('../Meta/DatasetBackup.csv', index=False)



=== SAMPLING DATA ===
Data tersedia: 1208 tweets
Target sample: 1000 tweets
Data final: 1000 tweets
Data berhasil disimpan ke '../Meta/DatasetBackup.csv'


In [12]:
# Statistik pembersihan data
print(f"\n=== STATISTIK PEMBERSIHAN ===")
print(f"Data awal: {len(df):,} tweets")
if len(cleaned_data) > 0:
    print(f"Data final: {len(cleaned_data):,} tweets")
    print(f"Persentase data yang dipertahankan: {(len(cleaned_data)/len(df)*100):.2f}%")
    print(f"Data yang dihapus: {len(df) - len(cleaned_data):,} tweets")
else:
    print("Tidak ada data yang memenuhi kriteria.")



=== STATISTIK PEMBERSIHAN ===
Data awal: 6,079 tweets
Data final: 1,208 tweets
Persentase data yang dipertahankan: 19.87%
Data yang dihapus: 4,871 tweets
